In [0]:
# CAMADA SILVER - Limpeza, padronizacao e regras de negocio
# Origem: tabelas da camada Bronze
# Nenhuma alteracao e realizada nas tabelas Bronze

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

print("Banco de dados 'silver' criado com sucesso.")

Banco de dados 'silver' criado com sucesso.


In [0]:
# silver.dim_consumidores
# Origem: bronze.tb_customers
# Regras: renomear colunas, upper case em cidade/estado e
# deduplicacao via Window Function pelo timestamp_ingestion


from pyspark.sql.functions import col, upper, row_number
from pyspark.sql.window import Window

df_customers = spark.table("bronze.tb_customers")

df_consumidores = df_customers.select(
    col("customer_id").alias("id_consumidor"),
    col("customer_unique_id").alias("id_consumidor_unico"),
    col("customer_name").alias("nome_consumidor"),
    col("customer_zip_code_prefix").alias("prefixo_cep"),
    col("customer_city").alias("cidade"),
    col("customer_state").alias("estado"),
    col("customer_gender").alias("genero"),
    col("customer_birth_date").alias("data_nascimento"),
    col("customer_age").alias("idade"),
    col("timestamp_ingestion")
)

df_consumidores = df_consumidores \
    .withColumn("cidade", upper(col("cidade"))) \
    .withColumn("estado", upper(col("estado")))

window_dedup = Window.partitionBy("id_consumidor").orderBy(col("timestamp_ingestion").desc())

df_consumidores = df_consumidores \
    .withColumn("row_num", row_number().over(window_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "timestamp_ingestion")

(df_consumidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_consumidores"))

print(f"Tabela silver.dim_consumidores criada com {df_consumidores.count()} registros.")

Tabela silver.dim_consumidores criada com 99441 registros.


In [0]:
# silver.fat_pedidos
# Origem: bronze.tb_orders
# Regras: traducao do status para portugues e criacao de
# colunas derivadas de tempo de entrega

from pyspark.sql.functions import col, when, datediff, to_timestamp

df_orders = spark.table("bronze.tb_orders")

df_pedidos = df_orders.withColumn("status",
    when(col("order_status") == "delivered",    "entregue")
    .when(col("order_status") == "canceled",    "cancelado")
    .when(col("order_status") == "shipped",     "enviado")
    .when(col("order_status") == "processing",  "processando")
    .when(col("order_status") == "invoiced",    "faturado")
    .when(col("order_status") == "unavailable", "indisponivel")
    .when(col("order_status") == "created",     "criado")
    .when(col("order_status") == "approved",    "aprovado")
    .otherwise(col("order_status"))
)

df_pedidos = df_pedidos \
    .withColumn("tempo_entrega_dias",
        datediff(
            to_timestamp(col("order_delivered_customer_date")),
            to_timestamp(col("order_purchase_timestamp"))
        )
    ) \
    .withColumn("tempo_entrega_estimado_dias",
        datediff(
            to_timestamp(col("order_estimated_delivery_date")),
            to_timestamp(col("order_purchase_timestamp"))
        )
    ) \
    .withColumn("diferenca_entrega_dias",
        col("tempo_entrega_dias") - col("tempo_entrega_estimado_dias")
    ) \
    .withColumn("entrega_no_prazo",
        when(col("order_status") != "delivered", "Nao Entregue")
        .when(col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Nao")
    )

df_pedidos = df_pedidos.select(
    col("order_id").alias("id_pedido"),
    col("customer_id").alias("id_consumidor"),
    col("status"),
    col("order_purchase_timestamp").alias("data_pedido"),
    col("order_approved_at").alias("data_aprovacao"),
    col("order_delivered_carrier_date").alias("data_entrega_transportadora"),
    col("order_delivered_customer_date").alias("data_entrega_cliente"),
    col("order_estimated_delivery_date").alias("data_entrega_estimada"),
    col("tempo_entrega_dias"),
    col("tempo_entrega_estimado_dias"),
    col("diferenca_entrega_dias"),
    col("entrega_no_prazo")
)

(df_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedidos"))

print(f"Tabela silver.fat_pedidos criada com {df_pedidos.count()} registros.")

Tabela silver.fat_pedidos criada com 99441 registros.


In [0]:

# silver.fat_itens_pedidos
# Origem: bronze.tb_order_items
# Regras: renomear colunas para portugues

from pyspark.sql.functions import col

df_order_items = spark.table("bronze.tb_order_items")

df_itens = df_order_items \
    .withColumn("id_pedido",   col("order_id")) \
    .withColumn("id_item",     col("order_item_id")) \
    .withColumn("id_produto",  col("product_id")) \
    .withColumn("id_vendedor", col("seller_id")) \
    .withColumn("preco_BRL",   col("price")) \
    .withColumn("preco_frete", col("freight_value")) \
    .select(
        col("id_pedido"),
        col("id_item"),
        col("id_produto"),
        col("id_vendedor"),
        col("shipping_limit_date").alias("data_limite_envio"),
        col("preco_BRL"),
        col("preco_frete")
    )

(df_itens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_itens_pedidos"))

print(f"Tabela silver.fat_itens_pedidos criada com {df_itens.count()} registros.")

Tabela silver.fat_itens_pedidos criada com 112650 registros.


In [0]:
# Camada Silver - fat_pagamentos_pedidos
# Origem: bronze.tb_order_payments
# Regras: traducao do tipo de pagamento para portugues

from pyspark.sql.functions import col, when

df_payments = spark.table("bronze.tb_order_payments")

df_pagamentos = df_payments \
    .withColumn("tipo_pagamento",
        when(col("payment_type") == "credit_card", "Cartao de Credito")
        .when(col("payment_type") == "boleto",     "Boleto")
        .when(col("payment_type") == "voucher",    "Voucher")
        .when(col("payment_type") == "debit_card", "Cartao de Debito")
        .when(col("payment_type") == "not_defined","Nao Definido")
        .otherwise(col("payment_type"))
    ) \
    .select(
        col("order_id").alias("id_pedido"),
        col("payment_sequential").alias("sequencial_pagamento"),
        col("tipo_pagamento"),
        col("payment_installments").alias("parcelas"),
        col("payment_value").alias("valor_pagamento")
    )

(df_pagamentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pagamentos_pedidos"))

print(f"Tabela silver.fat_pagamentos_pedidos criada com {df_pagamentos.count()} registros.")

Tabela silver.fat_pagamentos_pedidos criada com 103886 registros.


In [0]:
# Camada Silver - fat_avaliacoes_pedidos
# Origem: bronze.tb_order_reviews
# Regras: remover registros invalidos, tolerancia a falhas
# em datas com try_to_timestamp, e preenchimento de nulos
# em comentarios e titulos

from pyspark.sql.functions import col, when, current_timestamp, coalesce, lit, trim
from pyspark.sql.functions import try_to_timestamp

df_reviews = spark.table("bronze.tb_order_reviews")
df_pedidos_ids = spark.table("silver.fat_pedidos").select("id_pedido")

# Converte datas com tolerancia a falhas de formato
df_reviews = df_reviews \
    .withColumn("data_criacao_avaliacao",
        try_to_timestamp(col("review_creation_date"))) \
    .withColumn("data_resposta_avaliacao",
        try_to_timestamp(col("review_answer_timestamp")))

# Remove registros com pedido invalido
df_reviews = df_reviews.join(
    df_pedidos_ids,
    df_reviews["order_id"] == df_pedidos_ids["id_pedido"],
    "inner"
).drop("id_pedido")

# Remove registros com datas de comentario no futuro
df_reviews = df_reviews.filter(
    col("data_criacao_avaliacao") <= current_timestamp()
)

# Preenche titulos e comentarios vazios com texto padrao
df_avaliacoes = df_reviews \
    .withColumn("titulo_avaliacao",
        coalesce(
            when(trim(col("review_comment_title")) == "", None)
            .otherwise(col("review_comment_title")),
            lit("Sem titulo")
        )
    ) \
    .withColumn("comentario_avaliacao",
        coalesce(
            when(trim(col("review_comment_message")) == "", None)
            .otherwise(col("review_comment_message")),
            lit("Sem comentario")
        )
    ) \
    .select(
        col("review_id").alias("id_avaliacao"),
        col("order_id").alias("id_pedido"),
        col("review_score").alias("nota_avaliacao"),
        col("titulo_avaliacao"),
        col("comentario_avaliacao"),
        col("data_criacao_avaliacao"),
        col("data_resposta_avaliacao")
    )

(df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_avaliacoes_pedidos"))

print(f"Tabela silver.fat_avaliacoes_pedidos criada com {df_avaliacoes.count()} registros.")

Tabela silver.fat_avaliacoes_pedidos criada com 95307 registros.


In [0]:
# Camada Silver - dim_produtos
# Origem: bronze.tb_products
# Regras: renomear colunas, incluir nome_produto e deduplicacao
# via Window Function pelo timestamp_ingestion

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df_products = spark.table("bronze.tb_products")

window_dedup = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())

df_produtos = df_products \
    .withColumn("row_num", row_number().over(window_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "timestamp_ingestion") \
    .select(
        col("product_id").alias("id_produto"),
        col("product_name").alias("nome_produto"),
        col("product_category_name").alias("categoria_produto"),
        col("product_name_lenght").alias("tamanho_nome_produto"),
        col("product_description_lenght").alias("tamanho_descricao_produto"),
        col("product_photos_qty").alias("quantidade_fotos"),
        col("product_weight_g").alias("peso_produto_gramas"),
        col("product_length_cm").alias("comprimento_centimetros"),
        col("product_height_cm").alias("altura_centimetros"),
        col("product_width_cm").alias("largura_centimetros")
    )

(df_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_produtos"))

print(f"Tabela silver.dim_produtos criada com {df_produtos.count()} registros.")

Tabela silver.dim_produtos criada com 32951 registros.


In [0]:
# Camada Silver - dim_vendedores
# Origem: bronze.tb_sellers
# Regras: renomear colunas, incluir nome_vendedor, upper case
# em cidade/estado e deduplicacao via Window Function

from pyspark.sql.functions import col, upper, row_number
from pyspark.sql.window import Window

df_sellers = spark.table("bronze.tb_sellers")

window_dedup = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())

df_vendedores = df_sellers \
    .withColumn("row_num", row_number().over(window_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "timestamp_ingestion") \
    .withColumn("cidade", upper(col("seller_city"))) \
    .withColumn("estado", upper(col("seller_state"))) \
    .select(
        col("seller_id").alias("id_vendedor"),
        col("seller_name").alias("nome_vendedor"),
        col("seller_zip_code_prefix").alias("prefixo_cep"),
        col("cidade"),
        col("estado"),
        col("seller_registration_date").alias("data_registro")
    )

(df_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_vendedores"))

print(f"Tabela silver.dim_vendedores criada com {df_vendedores.count()} registros.")

Tabela silver.dim_vendedores criada com 3095 registros.


In [0]:
# Camada Silver - dim_categoria_produtos_traducao
# Origem: bronze.tb_product_category_name_translation
# Regras: renomear colunas para portugues e ingles

from pyspark.sql.functions import col

df_traducao = spark.table("bronze.tb_product_category_name_translation")

df_categoria_traducao = df_traducao \
    .drop("timestamp_ingestion") \
    .select(
        col("product_category_name").alias("nome_produto_pt"),
        col("product_category_name_english").alias("nome_produto_en")
    )

(df_categoria_traducao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_categoria_produtos_traducao"))

print(f"Tabela silver.dim_categoria_produtos_traducao criada com {df_categoria_traducao.count()} registros.")

Tabela silver.dim_categoria_produtos_traducao criada com 71 registros.


In [0]:
# Camada Silver - dim_cotacao_dolar
# Origem: bronze.tb_cotacao_dolar
# Regras: a API nao fornece cotacao para fins de semana.
# Cria-se um calendario continuo e os dias sem cotacao
# recebem o valor de fechamento da sexta-feira anterior
# usando last() com ignorenulls=True sobre Window Function

from pyspark.sql.functions import (
    col, to_date, avg, last,
    min as spark_min, max as spark_max
)
from pyspark.sql.window import Window

df_cotacao_raw = spark.table("bronze.tb_cotacao_dolar")

# Agrega por data (media do dia, pois pode haver multiplas cotacoes diarias)
df_cotacao_diaria = df_cotacao_raw \
    .withColumn("data", to_date(col("dataHoraCotacao"))) \
    .groupBy("data") \
    .agg(avg("cotacaoCompra").alias("cotacao_compra"))

# Obtém o intervalo de datas existente na cotacao
datas = df_cotacao_diaria.agg(
    spark_min("data").alias("data_min"),
    spark_max("data").alias("data_max")
).collect()[0]

# Cria calendario continuo cobrindo todo o periodo
df_calendario = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{datas.data_min}'),
        to_date('{datas.data_max}'),
        interval 1 day
    )) AS data
""")

# Join do calendario com as cotacoes reais
df_completo = df_calendario.join(df_cotacao_diaria, on="data", how="left")

# Preenche os nulos (fins de semana) com o ultimo valor disponivel
window_fill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)

df_cotacao_silver = df_completo \
    .withColumn("cotacao_compra", last(col("cotacao_compra"), ignorenulls=True).over(window_fill))

(df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_cotacao_dolar"))

print(f"Tabela silver.dim_cotacao_dolar criada com {df_cotacao_silver.count()} registros.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.dim_cotacao_dolar criada com 791 registros.


In [0]:
# Camada Silver - fat_pedido_total
# Juncao de fat_pedidos, fat_pagamentos_pedidos e
# dim_cotacao_dolar para gerar o valor total pago em
# BRL e USD por pedido

from pyspark.sql.functions import col, round as spark_round, to_date, sum as spark_sum

df_pedidos    = spark.table("silver.fat_pedidos")
df_pagamentos = spark.table("silver.fat_pagamentos_pedidos")
df_cotacao    = spark.table("silver.dim_cotacao_dolar")

# Agrega o valor total pago por pedido
df_valor_pedido = df_pagamentos.groupBy("id_pedido") \
    .agg(spark_sum("valor_pagamento").alias("valor_total_pago_brl"))

# Junta pedidos com valor total
df_total = df_pedidos.join(df_valor_pedido, on="id_pedido", how="left")

# Adiciona coluna de data sem hora para join com cotacao
df_total = df_total.withColumn("data_join", to_date(col("data_pedido")))

# Renomeia coluna de data na cotacao para evitar ambiguidade
df_cotacao_join = df_cotacao.withColumnRenamed("data", "data_cotacao")

# Junta com cotacao do dolar pela data do pedido
df_total = df_total.join(
    df_cotacao_join,
    df_total["data_join"] == df_cotacao_join["data_cotacao"],
    how="left"
).drop("data_join", "data_cotacao")

# Calcula valor em USD e arredonda para 2 casas decimais
df_pedido_total = df_total \
    .withColumn("valor_total_pago_brl", spark_round(col("valor_total_pago_brl"), 2)) \
    .withColumn("valor_total_pago_usd",
        spark_round(col("valor_total_pago_brl") / col("cotacao_compra"), 2)) \
    .select(
        col("id_pedido"),
        col("id_consumidor"),
        col("status"),
        col("valor_total_pago_brl"),
        col("valor_total_pago_usd"),
        col("data_pedido")
    )

(df_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedido_total"))

print(f"Tabela silver.fat_pedido_total criada com {df_pedido_total.count()} registros.")

Tabela silver.fat_pedido_total criada com 99441 registros.


In [0]:
# Otimizacao fisica das tabelas fato da camada Silver
# OPTIMIZE compacta os arquivos pequenos gerados pelo Delta
# ZORDER organiza os dados fisicamente pelas colunas de
# filtragem mais frequentes, acelerando as consultas

tabelas_otimizar = [
    ("silver.fat_pedido_total",       "id_pedido, data_pedido"),
    ("silver.fat_pedidos",            "id_pedido, data_pedido"),
    ("silver.fat_itens_pedidos",      "id_pedido"),
    ("silver.fat_pagamentos_pedidos", "id_pedido"),
    ("silver.fat_avaliacoes_pedidos", "id_pedido"),
]

for tabela, colunas in tabelas_otimizar:
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas})")
    print(f"Tabela {tabela} otimizada com ZORDER por ({colunas}).")

print("Otimizacao da camada Silver concluida.")

Tabela silver.fat_pedido_total otimizada com ZORDER por (id_pedido, data_pedido).
Tabela silver.fat_pedidos otimizada com ZORDER por (id_pedido, data_pedido).
Tabela silver.fat_itens_pedidos otimizada com ZORDER por (id_pedido).
Tabela silver.fat_pagamentos_pedidos otimizada com ZORDER por (id_pedido).
Tabela silver.fat_avaliacoes_pedidos otimizada com ZORDER por (id_pedido).
Otimizacao da camada Silver concluida.
